# One function, three IRs
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week3/Three_IRs_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 3, Module 2: Intermediate representations**  
**Author:** Dr. Kaoutar El Maghraoui

A compiler does not see your Python. It sees an intermediate representation (IR), and each IR level shows some details and hides others. FX keeps Python operations and the module structure. Core ATen spells every operation out in a small, typed operator set. The generated code is a set of kernels with their memory accesses. The lecture's rule: a pass can only change structure that the IR still holds.

In this notebook you take one small function, `relu(x @ w + b)`, through the three levels with PyTorch's own tools. Then you lower an attention block to Core ATen and see `Linear` disappear into lower-level operations. You finish by counting the memory traffic of the fused code against the lecture's fusion arithmetic.

**What you will do**
1. Capture `relu(x @ w + b)` as an FX graph.
2. Lower it to Core ATen and read the shape and dtype of every value.
3. Compile it with Inductor and read the generated code.
4. Lower an attention block to Core ATen.
5. Compare the memory traffic of the generated code with the three cases on the fusion slide.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about a minute.

## 1. FX: Python operations
The function is the lecture example: a matrix multiply, a bias add and a ReLU, with `x` of shape 512 by 768 and `w` of shape 768 by 3072, in FP32.

In [ ]:
import torch
import torch.fx

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)


def f(x, w, b):
    return torch.relu(x @ w + b)


x = torch.randn(512, 768, device=device)
w = torch.randn(768, 3072, device=device)
b = torch.randn(3072, device=device)

fx_graph = torch.fx.symbolic_trace(f)
print(fx_graph.graph)

**Reading the output**
- Three `call_function` nodes: `matmul`, `add` and `relu`, each naming the Python function it calls.
- There are no shapes, dtypes or devices. FX records which operations run and in which order, not the hardware details.

## 2. Core ATen: typed, decomposed operations
`make_fx` runs the function with fake tensors and records the ATen operators that PyTorch dispatches. With the Core ATen decomposition table, higher-level operators are rewritten into a smaller standard set. Every value now has a shape and a dtype.

In [ ]:
from torch.fx.experimental.proxy_tensor import make_fx
from torch._decomp import core_aten_decompositions

aten_graph = make_fx(f, decomposition_table=core_aten_decompositions())(x, w, b)
for node in aten_graph.graph.nodes:
    val = node.meta.get("val")
    typ = f"{str(val.dtype).replace('torch.', '')}{list(val.shape)}" if hasattr(val, "shape") else ""
    print(f"{node.op:14} {str(node.target):24} {typ}")

**Reading the output**
- `matmul` became `aten.mm.default`, `add` became `aten.add.Tensor`, and `relu` became `aten.relu.default`. Each name includes the overload, so the exact operator is resolved.
- Every value carries its dtype and shape, for example `float32[512, 3072]`. This is the information a backend needs to plan memory and kernels.
- The lecture shows the same function in HLO, where `relu` is written as a `maximum` with zero and the bias needs an explicit `broadcast`. The JAX demo in this folder prints it.

## 3. Generated code
`run_and_get_code` compiles the function with Inductor, runs it, and returns the source code Inductor wrote. The cell prints the lines that launch work and the fused Triton kernel.

In [ ]:
from torch._inductor.utils import run_and_get_code

compiled = torch.compile(f)
out, (code, *_) = run_and_get_code(compiled, x, w, b)
assert torch.allclose(out, f(x, w, b), rtol=1e-3, atol=1e-3)

lines = code.splitlines()
print("Lines in the generated file:", len(lines))
print("\nWhere work is launched:")
for line in lines:
    s = line.strip()
    if s.startswith("extern_kernels.") or (s.startswith("triton_") and ".run(" in s):
        print("   ", s[:110])

start = next((i for i, l in enumerate(lines) if l.startswith("def triton_")), None)
if start is not None:
    print("\nThe fused kernel:")
    for l in lines[start - 1: start + 20]:
        print("   ", l)

**Reading the output**
- `extern_kernels.mm` calls cuBLAS for the matrix multiply. Inductor does not generate matmuls when a tuned library is faster.
- One Triton kernel, with `fused_add_relu` or similar in its name, does the bias add and the ReLU in one pass. Inside it, look for `tl.load`, `tl.maximum` and `tl.store`: the ReLU became a `maximum` with zero, as in HLO.
- On a CPU runtime, Inductor writes C++ instead of Triton, and the kernel names differ.

## 4. An attention block in Core ATen
The lecture lowers an attention block written with `nn.Linear` layers. `aten.linear` is not in Core ATen, so it is decomposed again into a transpose and an `addmm` (a matrix multiply with a bias). The module names `q_proj`, `k_proj` and `v_proj` do not survive this step.

In [ ]:
import torch.nn as nn
from collections import Counter


class Attention(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.q_proj, self.k_proj, self.v_proj, self.o_proj = (nn.Linear(d, d) for _ in range(4))
        self.scale = d ** -0.5

    def forward(self, x):
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        p = torch.softmax((q @ k.transpose(-2, -1)) * self.scale, dim=-1)
        return self.o_proj(p @ v)


block = Attention().to(device).eval()
xa = torch.randn(1, 128, 256, device=device)
with torch.no_grad():
    attn_aten = make_fx(block, decomposition_table=core_aten_decompositions())(xa)

ops = Counter(str(n.target) for n in attn_aten.graph.nodes if n.op == "call_function")
print("Core ATen operators in the attention block:")
for name, count in ops.most_common():
    print(f"  {count:2d} x {name}")

**Reading the output**
- The four `Linear` layers became `aten.addmm` or `aten.mm` calls with `aten.t` or `aten.permute` transposes and `aten.view` reshapes. Decomposition happens in more than one step.
- Softmax may appear as `aten._softmax` or be split into `amax`, `sub`, `exp`, `sum` and `div`, depending on the PyTorch version.
- The module names are gone. The backend does not need them, but you will when you debug. The Week 5 provenance demo shows how PyTorch keeps a map back to the source.

## 5. Fusion arithmetic
The lecture counts how many times the 512 by 3072 FP32 output crosses the memory bus. The tensor is 6.29 MB, and there are three cases:

| Case | Crossings | Traffic |
|---|---|---|
| unfused: mm writes, add reads and writes, relu reads and writes | 5 | 31.5 MB |
| add and relu fused: mm writes, one kernel reads and writes | 3 | 18.9 MB |
| fused into the matmul epilogue: mm writes once | 1 | 6.29 MB |

The cell times eager mode (three kernels) and the compiled version, and prints which case the generated code matches.

In [ ]:
import time


def bench_ms(fn, iters=50):
    for _ in range(5):
        fn()
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3


out_mb = 512 * 3072 * 4 / 1e6
triton_kernels = sum(1 for l in lines if l.startswith("def triton_"))
crossings = 1 + 2 * triton_kernels      # mm writes once, each fused elementwise kernel reads and writes
print(f"Output tensor: {out_mb:.2f} MB")
print(f"Generated code: 1 cuBLAS matmul + {triton_kernels} Triton kernel(s) -> {crossings} crossings, {crossings * out_mb:.1f} MB")
print(f"Eager: {bench_ms(lambda: f(x, w, b)):.3f} ms per call (3 kernels, 5 crossings, {5 * out_mb:.1f} MB)")
print(f"Compiled: {bench_ms(lambda: compiled(x, w, b)):.3f} ms per call")

**Reading the output**
- The generated code matches the middle row: the matmul writes its output, and one fused kernel reads it and writes the result. That is 3 crossings instead of 5.
- The compiled version is faster by roughly the time it takes to move the saved 12.6 MB at the T4's bandwidth.
- Fusing the add and ReLU into the matmul itself, the bottom row, needs a matmul kernel with a custom epilogue. Inductor can do that with `max_autotune` and Triton matmul templates, on GPUs where those templates are supported.

## Try this
1. In section 2, remove `decomposition_table=...` and compare the operator names. Which ones are not in Core ATen?
2. In section 3, change `f` to `torch.relu(x @ w + b) * 2`. Does Inductor still use one Triton kernel?
3. In section 4, replace the softmax with `torch.nn.functional.gelu` on the scores and count the operators again.

## Summary
- FX holds Python operations and module calls, without shapes or layouts.
- Core ATen holds typed, decomposed operations from a small operator set. Higher-level operators such as `linear` are lowered further.
- The generated code holds kernels and memory accesses. Matmuls go to cuBLAS, and the operations around them are fused into Triton kernels.
- Each level makes new information visible to the passes that need it, and hides structure that the levels above had.

Lecture: Week 3, Module 2, slides 34 to 39 (FX to ATen to backend IR, IR levels, three IRs, seven lowering stages), and Module 3, slide 49 (arithmetic of fusion).